# Chapter 16 — The Vasicek and Gauss+ Models

教材 pp. 184–191；PDF 页序为教材页码加 11。

Gauss+ 采用教材三状态、两冲击的定义。表16.1给出的全样本参数可直接输入；原始国债样本未提供，不能声称重新估计了该表。

下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from gauss_plus_calculator import *

教材表16.1参数；p.189。价格边界及瞬时波动结构。

In [2]:
model=GaussPlus(1.0547,.6358,.0165,.01092,.00964,.212,.10555)
state=pd.DataFrame([[.02,.04,.06]],columns=['r','m','l'])
assert_allclose(model.discount(state,0),1)
assert_allclose(vasicek_half_life(.0165),42.01,atol=.01)
assert_allclose(model.diffusion[0],0)
_,cov=model.moments(state,1)
assert np.linalg.eigvalsh(cov).min()>=-1e-12
print(pd.DataFrame({'T':[1,5,10,30],'sigma':[model.yield_volatility(t) for t in [1,5,10,30]]}))

    T     sigma
0   1  0.003578
1   5  0.006368
2  10  0.007321
3  30  0.007237


模型生成曲线再反解状态；无波动常数状态给出指数折现。

In [3]:
tenors=np.array([.5,1,2,5,10,30])
curve=pd.DataFrame({'T':tenors,'yield':[-np.log(model.discount(state,t))/t for t in tenors]})
assert_allclose(model.fit_state(curve,'T','yield'),state,atol=1e-10)
deterministic=GaussPlus(1,.5,.1,0,0,0,.03)
flat=pd.DataFrame([[.03,.03,.03]],columns=['r','m','l'])
assert_allclose(deterministic.discount(flat,10),np.exp(-.3))

## Gauss+分阶段校准；pp.188–190（合成数据恢复，不是重估教材样本）

In [4]:
tenors=np.array([.5,1,3,5,7,15,20])
model=GaussPlus(1.0547,.6358,.0165,.01092,.00964,.212,.10555)
coeff=np.array([model.regression_loadings(t,[2,10]) for t in tenors])
data=pd.DataFrame({'T':tenors,'b2':coeff[:,0],'b10':coeff[:,1]})
speeds=GaussPlus.fit_speeds(data,'T',['b2','b10'],[2,10],[1.,.6,.02],([.001]*3,[3.]*3))
assert speeds.success
assert_allclose(speeds.x,[1.0547,.6358,.0165],atol=.001)
data['vol']=[model.yield_volatility(t) for t in tenors]
fit=GaussPlus.fit_volatilities(data,'T','vol',speeds.x,[.01,.01,.1],([.0001,.0001,-.99],[.1,.1,.99]))
assert fit.success
assert_allclose(fit.x,[.01092,.00964,.212],atol=.002)
state=pd.DataFrame([[.02,.04,.06]],columns=['r','m','l'])
data['yield']=[-np.log(model.discount(state,t))/t for t in tenors]
data['r']=.02
data['y2']=-np.log(model.discount(state,2))/2
data['y10']=-np.log(model.discount(state,10))/10
assert_allclose(model.fit_long_run_mean(data,'T','yield','r',['y2','y10'],[2,10]),.10555,atol=1e-7)